# PIKAN prediction for the infinite-domain problem

In [1]:
from pathlib import Path
import sys
from importlib import reload

import matplotlib.pyplot as plt
import numpy as np
import torch

# Find the repository utilities directory from either the notebook folder or workspace root.
notebook_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in [notebook_dir, *notebook_dir.parents] if (path / "utils").is_dir()),
    notebook_dir,
)
utilities_dir = repo_root / "utils"
if str(utilities_dir) not in sys.path:
    sys.path.insert(0, str(utilities_dir))

import infinite
import pinns_infinite
reload(infinite)
reload(pinns_infinite)

from infinite import analytical_solution_inf, coefficient_inf, evaluate_model_inf
from pinns_infinite import build_models_KAN, set_seed, train_dual_network

set_seed(2)
torch.set_default_dtype(torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## Tuned PIKAN configuration

In [ ]:
config = {
    # Choose "train" to fit a new model or "load" to use a stored checkpoint.
    "mode": "train",
    "checkpoint_name": "pikan_infinite_gaussian_weights.pt",
    "hidden_layers": 3,
    "hidden_units": 25,
    "grid_size": 5,
    "spline_order": 4,
    "adam_lr": 1e-2,
    "adam_iters": 2000,
    "lbfgs_iters": 2000,
    "sampling": "gaussian",
    "sigma": 5.5,
    "exp_scale": 1.0,
    "n_obs_u": 100,
    "n_obs_k": 100,
    "n_pde": 1000,
    "seed": 2,
    "pde_alpha": 0.5,
    "pde_beta": 5.0,
    "epsilon": 1.0,
}

for name, value in config.items():
    print(f"{name}: {value}")

mode: load
checkpoint_name: pikan_infinite_tuned_weights.pt
hidden_layers: 3
hidden_units: 25
grid_size: 5
spline_order: 4
adam_lr: 0.01
adam_iters: 2000
lbfgs_iters: 2000
sampling: gaussian
sigma: 5.5
exp_scale: 1.0
n_obs_u: 100
n_obs_k: 100
n_pde: 1000
seed: 2
pde_alpha: 0.5
pde_beta: 5.0
epsilon: 1.0


## Build the KAN models

In [3]:
model_u, model_k = build_models_KAN(
    device=device,
    hidden_layers=config["hidden_layers"],
    hidden_units=config["hidden_units"],
    grid_size=config["grid_size"],
    spline_order=config["spline_order"],
)

print(model_u)
print(model_k)

KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)
KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)


## Load the optimal PIKAN

In [4]:
results_dir = repo_root / "main" / "03_individual_prediction" / "results"
available_checkpoints = sorted(results_dir.glob("*.pt"))
print("Available stored models:")
for checkpoint in available_checkpoints:
    print(f"  - {checkpoint.name}")

mode = config["mode"].lower()
if mode not in {"train", "load"}:
    raise ValueError('config["mode"] must be either "train" or "load"')

if mode == "load":
    weights_path = results_dir / config["checkpoint_name"]
    if not weights_path.exists():
        raise FileNotFoundError(
            f"Stored checkpoint not found: {weights_path}. "
            f"Choose one of: {[path.name for path in available_checkpoints]}"
        )
    checkpoint = torch.load(weights_path, map_location=device)
    model_u.load_state_dict(checkpoint["model_u"])
    model_k.load_state_dict(checkpoint["model_k"])
    config.update(checkpoint.get("config", {}))
    metrics = checkpoint.get("metrics", {})
    print(f"Loaded stored model: {weights_path}")
else:
    history = train_dual_network(
        model_u,
        model_k,
        adam_lr=config["adam_lr"],
        adam_iters=config["adam_iters"],
        lbfgs_iters=config["lbfgs_iters"],
        verbose=True,
        print_every=100,
        save_every=100,
        lambda_pde_scheduler=True,
        adaptive_weights=True,
        alpha=7,
        update_every=100,
        regularization=False,
        sampling=config["sampling"],
        sigma=config["sigma"],
        exp_scale=config["exp_scale"],
        n_obs_u=config["n_obs_u"],
        n_obs_k=config["n_obs_k"],
        n_pde=config["n_pde"],
        seed=config["seed"],
        save_results=True,
        base_dir=str(results_dir),
        run_name="pikan_infinite_gaussian",
        pde_alpha=config["pde_alpha"],
        pde_beta=config["pde_beta"],
        epsilon=config["epsilon"],
        device=device,
    )
    weights_path = results_dir / config["checkpoint_name"]

model_u.eval()
model_k.eval()
print(model_u)
print(model_k)

Available stored models:
  - pikan_infinite_tuned_weights.pt
  - pikan_semi_infinite_tuned_weights.pt
  - pikan_semi_infinite_uniform_weights.pt
Loaded stored model: /home/orincon/unbounded-domains/main/03_individual_prediction/results/pikan_infinite_tuned_weights.pt
KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)
KAN(
  (layers): ModuleList(
    (0-3): 4 x KANLinear(
      (base_activation): SiLU()
    )
  )
)


## Evaluate against the analytical solution

Compute global, in-domain, and out-of-domain mean absolute errors for both learned fields.

In [5]:
evaluation = evaluate_model_inf(
    model_u=model_u,
    model_k=model_k,
    analytical_solution=analytical_solution_inf,
    coefficient=coefficient_inf,
    sampling=config["sampling"],
    train_xmin=-5.0,
    train_xmax=5.0,
    train_ymin=-5.0,
    train_ymax=5.0,
    eval_xmin=-8.0,
    eval_xmax=8.0,
    eval_ymin=-8.0,
    eval_ymax=8.0,
    n_grid=400,
    alpha=config["pde_alpha"],
    beta=config["pde_beta"],
    epsilon=config["epsilon"],
    device=device,
    verbose=True,
)

metric_names = [
    "err_u_global", "err_k_global",
    "err_u_inside", "err_k_inside",
    "err_u_outside", "err_k_outside",
]
metrics = {name: float(evaluation[name]) for name in metric_names}

if mode == "train":
    torch.save(
        {
            "model_u": model_u.state_dict(),
            "model_k": model_k.state_dict(),
            "config": config,
            "metrics": metrics,
        },
        weights_path,
    )
    print(f"Saved trained model: {weights_path}")

metrics


Spatial generalization (MAE)
Training domain : [-5.0, 5.0] × [-5.0, 5.0]
Evaluation domain : [-8.0, 8.0] × [-8.0, 8.0]

Global MAE
u : 1.099e-03
k : 1.072e-03

Inside training domain
u : 1.441e-03
k : 7.301e-04

Outside training domain
u : 8.795e-04
k : 1.292e-03


{'err_u_global': 0.001098704766821438,
 'err_k_global': 0.001072381301849149,
 'err_u_inside': 0.0014406407356177093,
 'err_k_inside': 0.0007301386591252567,
 'err_u_outside': 0.0008795150432340848,
 'err_k_outside': 0.0012917676112875417}